In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
print("NumPy:", np.__version__)

In [ ]:
x, y = 1.0, 1.0
w, b, v = 0.5, 0.0, 1.0
z = w*x + b
h = sigmoid(z)
y_hat = v*h
loss = 0.5*(y_hat-y)**2
print(f"z={z:.4f}, h={h:.4f}, prediction={y_hat:.4f}, loss={loss:.6f}")

In [ ]:
dL_dyhat = y_hat - y
dyhat_dh = v
dh_dz = h*(1-h)
dz_dw = x
grad = dL_dyhat*dyhat_dh*dh_dz*dz_dw
def scalar_loss(weight):
    return 0.5*(v*sigmoid(weight*x+b)-y)**2
eps = 1e-5
numeric_grad = (scalar_loss(w+eps)-scalar_loss(w-eps))/(2*eps)
print("four factors:", dL_dyhat, dyhat_dh, dh_dz, dz_dw)
print(f"chain rule={grad:.6f}, numerical={numeric_grad:.6f}")
assert np.isclose(grad, numeric_grad, atol=1e-8)

In [ ]:
lr = 0.5
w_new = w - lr*grad
print(f"w: {w:.4f} -> {w_new:.4f}")
print(f"loss: {scalar_loss(w):.6f} -> {scalar_loss(w_new):.6f}")
weights = np.linspace(-3, 5, 200)
plt.figure(figsize=(7, 4))
plt.plot(weights, [scalar_loss(t) for t in weights])
plt.scatter([w, w_new], [scalar_loss(w), scalar_loss(w_new)], c=['#de6c38','#007f86'], s=70)
plt.xlabel('Weight w'); plt.ylabel('Loss'); plt.title('One gradient descent step')
plt.grid(alpha=.2); plt.show()

In [ ]:
X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)
def init_params(seed=7):
    rng = np.random.default_rng(seed)
    return {'W1': rng.normal(0, 1, (2,4)), 'b1': np.zeros((1,4)),
            'W2': rng.normal(0, 1, (4,1)), 'b2': np.zeros((1,1))}
print(np.column_stack([X,Y]))
print({k: v.shape for k,v in init_params().items()})

In [ ]:
def forward(X, p):
    H = sigmoid(X @ p['W1'] + p['b1'])
    Z2 = H @ p['W2'] + p['b2']
    P = sigmoid(Z2)
    return H, Z2, P
def bce(Z2, Y):
    return np.mean(np.logaddexp(0, Z2) - Y*Z2)
p0 = init_params()
H, Z2, P = forward(X, p0)
print("initial probabilities:", P.ravel())
print("initial loss:", bce(Z2, Y))

In [ ]:
def backward(X, Y, H, P, p):
    dZ2 = (P-Y)/len(X)
    dW2 = H.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)
    dH = dZ2 @ p['W2'].T
    dZ1 = dH*H*(1-H)
    return {'W1': X.T @ dZ1, 'b1': dZ1.sum(axis=0, keepdims=True),
            'W2': dW2, 'b2': db2}
def train(lr=1.0, epochs=5000, seed=7):
    p = init_params(seed)
    losses = []
    for epoch in range(epochs):
        H, Z2, P = forward(X, p)
        losses.append(float(bce(Z2, Y)))
        grads = backward(X, Y, H, P, p)
        for name in p:
            p[name] -= lr*grads[name]
    _, Z2, P = forward(X, p)
    losses.append(float(bce(Z2, Y)))
    return p, np.array(losses), P
p, losses, P = train()
print("initial / final loss:", losses[0], losses[-1])
print("x1 x2 / answer / probability / prediction")
print(np.column_stack([X, Y, P, (P>=0.5).astype(int)]))
assert np.all((P>=0.5)==Y)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(losses, color='#007f86')
axes[0].set(xlabel='Epoch', ylabel='BCE loss', title='Training loss')
grid = np.linspace(-.3,1.3,180)
xx, yy = np.meshgrid(grid,grid)
_, _, pp = forward(np.column_stack([xx.ravel(), yy.ravel()]), p)
zz = pp.reshape(xx.shape)
im = axes[1].contourf(xx, yy, zz, levels=np.linspace(0,1,21), cmap='coolwarm',vmin=0,vmax=1)
axes[1].contour(xx, yy, zz, levels=[.5], colors='black')
axes[1].scatter(X[:,0],X[:,1], c=Y.ravel(),cmap='coolwarm',vmin=0,vmax=1,edgecolors='black',s=100)
axes[1].set(xlabel='x1',ylabel='x2',title='XOR probability and boundary')
fig.colorbar(im, ax=axes[1],label='P(y=1)'); plt.tight_layout(); plt.show()

In [ ]:
rates = [0.01, 1.0, 200.0]
plt.figure(figsize=(8,4))
for rate in rates:
    _, history, prob = train(lr=rate)
    accuracy = np.mean((prob>=.5)==Y)
    print(f"lr={rate:5.2f}, final loss={history[-1]:.6f}, training accuracy={accuracy:.0%}")
    plt.plot(history, label=f'lr={rate}')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('BCE loss (log scale)')
plt.title('Same initialization, different learning rates'); plt.legend(); plt.grid(alpha=.2); plt.show()